[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C08_Training_Systems_Course/02_mixed_precision_checkpointing/02_mixed_precision_checkpointing.ipynb)

# 02 · 混合精度与重计算

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在真实模型规模上算精度格式省多少显存、模拟 fp16 下溢与 loss scaling、验证梯度累积、算重计算的算力/显存交易。

**你将完成：**
1. fp32/fp16/bf16/fp8 的字节账与动态范围
2. 模拟 fp16 下溢，实现动态 loss scaling
3. 验证梯度累积 == 大 batch（逐元素相等）
4. activation checkpointing 的 √L 显存 vs 1.33× 算力

> 数据：真实 Pythia config（算显存账）。

## 0 · config 管线

In [ ]:
import os, json, urllib.request
import numpy as np
CACHE=os.path.expanduser("~/.training_systems_data"); os.makedirs(CACHE, exist_ok=True)
MODELS={"pythia-1.4b":"https://huggingface.co/EleutherAI/pythia-1.4b/resolve/main/config.json",
        "pythia-6.9b":"https://huggingface.co/EleutherAI/pythia-6.9b/resolve/main/config.json",
        "pythia-12b":"https://huggingface.co/EleutherAI/pythia-12b/resolve/main/config.json"}
def load_config(m):
    p=os.path.join(CACHE,f"{m}.json")
    if not os.path.exists(p): urllib.request.urlretrieve(MODELS[m],p)
    c=json.load(open(p)); g=lambda *k: next(c[x] for x in k if x in c); h=g("hidden_size","n_embd")
    return dict(L=g("num_hidden_layers","n_layer"),h=h,heads=g("num_attention_heads","n_head"),
                V=g("vocab_size"),I=c.get("intermediate_size",4*h))
def param_count(c):
    L,h,V,I=c["L"],c["h"],c["V"],c["I"]; return 2*V*h+L*(4*h*h+4*h+2*h*I+(I+h)+4*h)+h
GB=1024**3
print('ok')

## 1 · 浮点格式的字节账与范围

16 位格式省一半显存。fp16 范围窄（~6e4），bf16 范围同 fp32（~3e38）。

In [ ]:
DTYPE_BYTES={"fp32":4,"fp16":2,"bf16":2,"fp8":1}
RANGE={"fp32":3.4e38,"fp16":65504,"bf16":3.4e38,"fp8":448}
P=param_count(load_config("pythia-6.9b"))
print("pythia-6.9b 仅权重显存:")
for d,b in DTYPE_BYTES.items():
    print(f"  {d:5s} {b}字节/参数  {P*b/GB:6.1f} GB   最大可表示≈{RANGE[d]:.0e}")
print("\n=> fp16 与 bf16 都省一半显存，但 fp16 最大才 6.5e4，大梯度会溢出")

## 2 · 模拟 fp16 下溢 + 动态 loss scaling

fp16 表示不了太小的数（下溢成 0）。loss scaling 把梯度放大进可表示区间。

In [ ]:
def to_fp16(x):  # 用 numpy float16 模拟
    return np.float16(x)
tiny = 1e-7
print(f"真实小梯度 {tiny} 存成 fp16 = {float(to_fp16(tiny))}  (下溢为 0! 参数不更新)")
scale = 1024.0
print(f"放大 {scale}x 后存 fp16 = {float(to_fp16(tiny*scale))}  (可表示)，更新前再除回")

def dynamic_loss_scale(grad_max_seq, init=2**16):
    "grad_max_seq: 每步未缩放梯度最大值序列。返回每步采用的 scale。"
    s=init; out=[]
    for gmax in grad_max_seq:
        while gmax*s > 65504 and s>1:    # 会溢出 -> 减半跳过
            s/=2
        out.append(s); s*=2              # 正常 -> 尝试加倍
    return out
scales=dynamic_loss_scale([1e-4,1e-4,1e-2,1e-4])
print("动态 scale 序列:", scales, "(遇到大梯度自动降)")

## 3 · 梯度累积 == 大 batch

把大 batch 拆成 k 个 micro-batch，累加梯度。验证与一次性大 batch 梯度逐元素相等。

In [ ]:
rng=np.random.default_rng(0)
# 玩具线性回归 loss=0.5*mean((Xw-y)^2)，梯度 = X^T(Xw-y)/n
X=rng.normal(size=(64,5)); y=rng.normal(size=64); w=rng.normal(size=5)
def grad(Xb,yb): return Xb.T@(Xb@w-yb)/len(yb)

full = grad(X,y)                                  # 一次性 batch=64
k=4; micro=np.array_split(np.arange(64),k)
accum=np.zeros(5)
for idx in micro:
    accum += grad(X[idx],y[idx]) * (len(idx)/64)  # 按样本占比加权
print("大batch梯度:", full.round(5))
print("累积梯度  :", accum.round(5))
print("逐元素相等:", np.allclose(full, accum), "✓ (梯度累积数学上等价于大batch)")

## 4 · activation checkpointing：√L 显存换 1.33× 算力

只存 √L 个 checkpoint，反向重算。在真实 pythia-12b 上算显存节省与算力代价。

In [ ]:
cfg=load_config("pythia-12b"); L=cfg["L"]
def act_full(L): return L            # 正比于层数（单位化）
def act_ckpt(L): return int(np.ceil(np.sqrt(L)))  # 只存 √L 个
print(f"pythia-12b L={L} 层")
print(f"普通: 存 {act_full(L)} 份 activation")
print(f"重计算: 存 {act_ckpt(L)} 份  -> 显存降到 {act_ckpt(L)/act_full(L):.1%}")
print(f"算力代价: 反向多一次前向 -> 约 +{1/3:.0%} 前向 FLOPs (3 次前向等价 vs 2 次)")
print("\n=> 长上下文/深模型训练靠它把 activation 显存压下来")

## 5 · bf16 vs fp16 的下溢临界点

同是 16 位、同样省一半显存，但 fp16 只有 5 位指数（最小正规数 ≈ 6e-5，靠 subnormal 勉强撑到 ≈6e-8），bf16 有 8 位指数（动态范围≈fp32）。下面在一串越来越小的梯度上看谁先 flush 到零，直观理解 **为什么 bf16 训练无需 loss scaling，而 fp16 必须靠它把梯度放大进可表示区**。

In [ ]:
# bf16 vs fp16 在小数处的下溢临界：同样 16 位，谁先归零？
# fp16: 5 位指数，最小正规数 ≈ 6.1e-5，再靠 subnormal 撑到 ≈6e-8，更小就 flush 到 0。
# bf16: 8 位指数，动态范围≈fp32，这些小数照样表示。
def to_bf16(x):
    # bf16 = fp32 截断低 16 位尾数。用 float32 的 bit 表示模拟
    b = np.float32(x).view(np.uint32)
    b = (b + 0x8000) & 0xFFFF0000          # round-to-nearest 后截断
    return np.float32(np.uint32(b).view(np.float32))

grads = [1e-4, 1e-6, 1e-7, 1e-8, 1e-9]
print(f"{'真实梯度':>10s} {'-> fp16':>14s} {'-> bf16':>14s}")
for g in grads:
    f16=float(np.float16(g)); b16=float(to_bf16(g))
    print(f"{g:10.0e} {f16:14.2e} {b16:14.2e}")
# 自检：1e-8 在 fp16 下溢为 0，但 bf16 仍非零（范围宽）
assert float(np.float16(1e-8)) == 0.0, "1e-8 已超出 fp16 subnormal 下界，flush 到 0"
assert float(to_bf16(1e-8)) > 0.0, "bf16 范围宽，1e-8 不下溢"
# loss scaling 把 fp16 下溢的 1e-8 救回：×2^16 后落入可表示区
assert float(np.float16(1e-8 * 2**16)) > 0.0, "放大后 fp16 可表示"
print("\n=> fp16 在 1e-8 处已下溢归零(梯度丢失、参数不更新)，bf16 因 8 位指数照样表示；")
print("   这就是 bf16 训练无需 loss scaling、而 fp16 必须靠它(把梯度放大进可表示区)续命的根因")

---
## ✏️ 练习区

### ✏️ 练习 1：精度格式的显存节省

实现 `weight_memory_gb(P, dtype)` 用 `DTYPE_BYTES`，和 `savings_vs_fp32(P, dtype)` 返回相对 fp32 省的比例。

In [ ]:
def weight_memory_gb(P, dtype):
    # TODO: P * DTYPE_BYTES[dtype] / GB
    raise NotImplementedError
def savings_vs_fp32(P, dtype):
    # TODO: 1 - bytes[dtype]/bytes['fp32']
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测 ——
P=param_count(load_config("pythia-6.9b"))
assert abs(weight_memory_gb(P,"fp16") - P*2/GB) < 1e-6
assert abs(savings_vs_fp32(P,"bf16") - 0.5) < 1e-9
assert abs(savings_vs_fp32(P,"fp8") - 0.75) < 1e-9
print(f"练习 1 通过 ✓  6.9b: fp32={weight_memory_gb(P,'fp32'):.0f}G -> fp16={weight_memory_gb(P,'fp16'):.0f}G")


### ✏️ 练习 2：会不会 fp16 溢出 + 选 loss scale

实现 `fp16_overflows(x)`（|x|>65504 即溢出）和 `pick_loss_scale(grad_max)`：
返回不让 `grad_max*scale` 溢出的最大 2 的幂。

In [ ]:
def fp16_overflows(x):
    # TODO: abs(x) > 65504
    raise NotImplementedError
def pick_loss_scale(grad_max):
    # TODO: 最大的 2^k 使 grad_max*2^k <= 65504
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测 ——
assert fp16_overflows(1e5) and not fp16_overflows(1e4)
s=pick_loss_scale(1e-3)
assert s*1e-3 <= 65504 and (s*2)*1e-3 > 65504
assert not fp16_overflows(1e-3*s)
print(f"练习 2 通过 ✓  grad_max=1e-3 -> loss scale={s:.0f}")


### ✏️ 练习 3：梯度累积等价

实现 `accumulate_grad(X, y, w, k)`：把数据分 k 份，按样本占比加权累加梯度，返回累积梯度。
应等于全 batch 梯度。

In [ ]:
def accumulate_grad(X, y, w, k):
    # TODO: split 成 k 份，sum( grad(part)*(n_part/n_total) )
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
rng=np.random.default_rng(1); X=rng.normal(size=(60,4)); y=rng.normal(size=60); w=rng.normal(size=4)
full=X.T@(X@w-y)/60
for k in [2,3,5,6]:
    assert np.allclose(accumulate_grad(X,y,w,k), full), f"k={k} 应等价"
print("练习 3 通过 ✓  任意 k 的累积梯度都 == 大 batch 梯度")


### ✏️ 练习 4：重计算的显存/算力交易

实现 `checkpoint_tradeoff(L)`：返回 `(stored_fraction, extra_forward_ratio)`，
其中存 √L 份、额外算力约为再做一次前向（用 1/3 近似 3前向vs2前向 的额外比例）。

In [ ]:
def checkpoint_tradeoff(L):
    # TODO: stored = ceil(sqrt(L))/L ; extra = 1/3
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
frac, extra = checkpoint_tradeoff(36)   # pythia-12b
assert frac < 0.25, "36 层只存 6 份，<25%"
assert abs(extra - 1/3) < 1e-9
# 层越多，节省越夸张
assert checkpoint_tradeoff(100)[0] < checkpoint_tradeoff(36)[0]
print(f"练习 4 通过 ✓  L=36: 只存 {frac:.0%} activation, 代价 +{extra:.0%} 算力")


---
## 📖 参考答案

In [ ]:
# 练习 1
def weight_memory_gb(P, dtype): return P*DTYPE_BYTES[dtype]/GB
def savings_vs_fp32(P, dtype): return 1 - DTYPE_BYTES[dtype]/DTYPE_BYTES["fp32"]
print("练习 1 ✓")

In [ ]:
# 练习 2
def fp16_overflows(x): return abs(x) > 65504
def pick_loss_scale(grad_max):
    s=1.0
    while grad_max*s*2 <= 65504: s*=2
    return s
print("练习 2 ✓")

In [ ]:
# 练习 3
def accumulate_grad(X, y, w, k):
    n=len(y); acc=np.zeros_like(w)
    for idx in np.array_split(np.arange(n), k):
        acc += X[idx].T@(X[idx]@w-y[idx])/len(idx) * (len(idx)/n)
    return acc
print("练习 3 ✓")

In [ ]:
# 练习 4
def checkpoint_tradeoff(L):
    return int(np.ceil(np.sqrt(L)))/L, 1/3
print("练习 4 ✓ —— 显存/算力/精度三角交易是 systems 的核心直觉")